## **Data Sources**



two datasets were utilized, with three playing a direct role in constructing the customer master table:




| Dataset Name                 | Description                                          | Role                                |
| ---------------------------- | ---------------------------------------------------- | ----------------------------------- |
| bank_transactions_data_2.csv | Bank transfer logs                                   | it will be use for the completed baseline of transaction data|
| CUSTOMER_MASTER_FULL.csv     | Customer data                                        | use some fields here for connecting transaction and customer info, like customer_id, balance, card ownership (it will be use for the baseline of transaction data)                  |

note: for the customer master data will be the baseline but only some information that will be used, then will completed by adding information transaction from bank_transactions_data_2.csv 


In [1]:
import pandas as pd
import numpy as np

In [2]:
csv = {"customer": r"D:\KULIAH\SEMESTER 4\DRAFT\Step by step\Final Source Data\CUSTOMER_MASTER_FULL.csv",
    "bank_trx": r"D:\KULIAH\SEMESTER 4\DRAFT\Step by step\Final Source Data\data raw\bank_transactions_data_2.csv",
}

In [3]:
def load_csvs():
    data = {}   # important!
    for key, path in csv.items():

        # detect delimiter based on file name
        if "customer" in path:
            sep = ';'
       ## elif "marketing" in path:   # <-- your marketing_campaign.tsv
            sep = '\t'
        else:
            sep = ','

        print(f"Loading {path} using sep='{sep}' ...")
        df = pd.read_csv(path, sep=sep)

        print(f"  → Loaded {len(df):,} rows, {len(df.columns)} columns")
        data[key] = df

    return data


In [4]:
data = load_csvs()

Loading D:\KULIAH\SEMESTER 4\DRAFT\Step by step\Final Source Data\CUSTOMER_MASTER_FULL.csv using sep=',' ...
  → Loaded 6,121 rows, 20 columns
Loading D:\KULIAH\SEMESTER 4\DRAFT\Step by step\Final Source Data\data raw\bank_transactions_data_2.csv using sep=',' ...
  → Loaded 2,512 rows, 16 columns


## **TRANSACTION_MASTER_FULL – ETL Data Dictionary**


**Key assumptions:**

- The simulated Fictitious bank (Bank ABC) is based in Estonia.

- Customers have various citizenships (as reflected in the customer master), but their accounts are based in Estonia.

- For AML simulation, about 20% of incoming transfers originate from watchlist/high-risk jurisdictions.

| Column                            | Type     | Source                        | Description                                                                                            |
| --------------------------------- | -------- | ----------------------------- | ------------------------------------------------------------------------------------------------------ |
| **transaction_id**                | string   | Bank transactions (baseline)  | Unique identifier for each transaction. Taken from the original transaction dataset.                   |
| **transaction_date**              | datetime | Bank transactions             | Timestamp of the transaction. Parsed and standardized into datetime format.                            |
| **transaction_amount**            | float    | Bank transactions             | Monetary amount of the transaction. Positive numeric value.                                            |
| **transaction_type**              | string   | Bank transactions             | Transaction type from baseline dataset (e.g., TRANSFER, PAYMENT, CASH_WITHDRAWAL).                     |
| **transaction_duration**          | float    | Bank transactions / synthetic | Duration of transaction session when available; otherwise synthetically assigned.                      |
| **login_attempts**                | int      | Synthetic                     | Number of login attempts prior to the transaction. Your code generates this field.                     |
| **previous_transaction_date**     | datetime | Derived                       | Previous transaction timestamp for the same customer. Calculated using a per-customer sorted window.   |
| **previous_transaction_date_raw** | datetime | Derived                       | Raw previous transaction timestamp before cleaning/adjustment.                                         |
| **location_device**               | string   | Bank transactions (renamed)   | Device or terminal location (ATM, POS terminal, browser environment). Derived from `Location` column.  |
| **device_id**                     | string   | Bank transactions             | Device or terminal identifier used for the transaction.                                                |
| **ip_address**                    | string   | Bank transactions             | IP address associated with online or mobile transactions.                                              |
| **channel**                       | string   | Standardized from baseline    | Transaction delivery channel (ATM, WEB, MOBILE, POS, etc.). Normalized in code.                        |
| **location**                      | string   | Bank transactions             | General physical location of the transaction; kept as a separate field from `location_device`.         |
| **customer_id**                   | string   | CUSTOMER_MASTER_FULL          | FK to customer master dataset. Merged into the transaction dataset using joins.                        |
| **first_name**                    | string   | CUSTOMER_MASTER_FULL          | Customer first name carried through during merge; used for identification or AML simulations.          |
| **surname**                       | string   | CUSTOMER_MASTER_FULL          | Customer surname synced from customer master dataset.                                                  |
| **dt_customer**                   | date     | CUSTOMER_MASTER_FULL          | Customer onboarding date. Used for ensuring temporal validity in transaction generation.               |
| **account_id**                    | string   | Derived                       | Synthetic account identifier generated in your code, used to link customer → account → transactions.   |
| **account_type**                  | string   | Synthetic                     | Randomly assigned (e.g., CREDIT, DEBIT, SAVINGS) based on your generation logic.                       |
| **currency**                      | string   | Synthetic                     | Transaction currency (e.g., EUR). Default is EUR in your code unless modified.                         |
| **account_created_date**          | date     | Synthetic                     | Synthetic account-opening date generated before transaction activity.                                  |
| **sender_account**                | string   | Synthetic                     | Account number of the sender. Generated using controlled distribution logic.                           |
| **sender_country**                | string   | Synthetic                     | Country of the sender, derived using your synthetic rules for INTERNAL/DOMESTIC/INTERNATIONAL routing. |
| **sender_region**                 | string   | Synthetic                     | Geographic region of the sender based on country mapping.                                              |
| **receiver_country**              | string   | Synthetic                     | Country of the transaction’s receiver, using the same synthetic routing distributions.                 |
| **receiver_region**               | string   | Synthetic                     | Geographic region of the receiver.                                                                     |


## **ETL PIPELINE OVERVIEW**

The ETL pipeline consists of eight major steps:

1.Extraction

    - Load transaction data from bank_transactions_data_2.csv.
    
    - Load customer data from CUSTOMER_MASTER_FULL.csv for enrichment.

2. Standardization & Normalization

    - Rename raw transaction columns into a consistent schema.
    
    - Convert all column names to lowercase.
    
    - Normalize categorical values such as channel.

3. Feature Engineering

    - Generate previous_transaction_date and previous_transaction_date_raw per customer.
    
    - Create synthetic behavioral features: login_attempts and transaction_duration.

4. Customer Enrichment

    - Merge transaction data with customer master using customer_id.
    
    - Add first_name, surname, and dt_customer to each transaction record.

5. Account & Routing Generation

    - Generate synthetic account attributes: account_id, account_type, currency, account_created_date.
    
    - Create sender/receiver routing fields: sender_account, sender_country, sender_region, receiver_country, receiver_region.

6. Canonical Column Selection

    Select only the cleaned, standardized fields for the final transaction table.

7. Export

    - Export the final dataset as TRANSACTION_MASTER_FULL.csv.
    
    - load into SQL Server for GRC, AML, and audit simulations (if needed).
      
**A Data lineage is illustrated below:**

## **DATA LINEAGE DIAGRAM (Customer Data & Transaction Data)**

```mermaid

flowchart LR
    A["bank_transactions_data_2.csv\n(Raw Transaction Data)"] --> B[Standardize & Rename]
    C["CUSTOMER_MASTER_FULL.csv\n(Customer Master Data)"] --> E[Join on customer_id]

    B --> D["Behavioral Features\n(previous_transaction_date, login_attempts, duration)"]
    D --> E

    E --> F["Account & Routing Generation\n(account_id, sender/receiver country & region)"]

    F --> G["TRANSACTION_MASTER_FULL.csv\n(Final Transaction Dataset)"]

  
```


## **ERD DIAGRAM**

``` mermaid

erDiagram

    CUSTOMER_MASTER_FULL {
        string customer_id
        string first_name
        string surname
        string gender
        date date_of_birth
        int age
        string marital
        string education
        string job
        float income
        string contact
        string nationality
        float balance
        int has_cr_card
        string default
        string housing
        string loan
        int credit_score
        int tenure
        date dt_customer
    }

    TRANSACTION_MASTER_FULL {
        string transaction_id
        datetime transaction_date
        float transaction_amount
        string transaction_type
        float transaction_duration
        int login_attempts
        datetime previous_transaction_date
        datetime previous_transaction_date_raw
        string location_device
        string device_id
        string ip_address
        string channel
        string location
        string customer_id
        string first_name
        string surname
        date dt_customer
        string account_id
        string account_type
        string currency
        date account_created_date
        string sender_account
        string sender_country
        string sender_region
        string receiver_country
        string receiver_region
    }

    CUSTOMER_MASTER_FULL ||--o{ TRANSACTION_MASTER_FULL : "has transactions"




```

## **SQL SCHEMA — TRANSACTION_MASTER**

```
CREATE TABLE TRANSACTION_MASTER_FULL (
    transaction_id              VARCHAR(50)     NOT NULL PRIMARY KEY,
    transaction_date            DATETIME        NULL,
    transaction_amount          DECIMAL(18,2)   NULL,
    transaction_type            VARCHAR(50)     NULL,
    transaction_duration        FLOAT           NULL,
    login_attempts              INT             NULL,
    previous_transaction_date   DATETIME        NULL,
    previous_transaction_date_raw DATETIME      NULL,
    location_device             VARCHAR(100)    NULL,
    device_id                   VARCHAR(100)    NULL,
    ip_address                  VARCHAR(45)     NULL,
    channel                     VARCHAR(20)     NULL,
    location                    VARCHAR(100)    NULL,
    customer_id                 VARCHAR(50)     NULL,
    first_name                  VARCHAR(100)    NULL,
    surname                     VARCHAR(100)    NULL,
    dt_customer                 DATE            NULL,
    account_id                  VARCHAR(50)     NULL,
    account_type                VARCHAR(20)     NULL,
    currency                    VARCHAR(10)     NULL,
    account_created_date        DATE            NULL,
    sender_account              VARCHAR(50)     NULL,
    sender_country              VARCHAR(50)     NULL,
    sender_region               VARCHAR(50)     NULL,
    receiver_country            VARCHAR(50)     NULL,
    receiver_region             VARCHAR(50)     NULL,

    CONSTRAINT FK_TRANSACTION_CUSTOMER
        FOREIGN KEY (customer_id)
        REFERENCES CUSTOMER_MASTER_FULL(customer_id)
);


```

### **Extract Phase**

In this phase, all raw datasets are imported into the ETL workflow.
The primary baseline dataset is bank_transactions_data_2.csv, which contains the core transactional attributes needed to generate a synthetic transaction log (e.g., transaction ID, date, amount, type, channel, device, IP address, and location).

To make the transaction data meaningful and internally consistent, the pipeline also loads:

CUSTOMER_MASTER_FULL.csv → used to link each transaction to an internal customer through customer_id, and to enrich transactions with first_name, surname, and dt_customer.

### **Bank Transfer Data as a Baseline**

this dataset contains minimal information I need for developing transactional data like real data in the bank, financial services, and fintech but essential. It includes:

- transaction identifiers

- timestamps

- transaction amounts and types

- device and IP information

- transaction location and channel

This dataset serves as the backbone of the ETL workflow.
Additional attributes—such as behavioral features, customer identity fields, synthetic account data, and routing logic—are generated or enriched later in the pipeline to create a complete transaction model ready for AML, risk, and audit simulation.


In [5]:
df_bank_trx = data["bank_trx"] # Baseline data
df_customer = data["customer"] # additional data

In [6]:
df_bank_trx.head()

,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21,2024-11-04 08:08:08
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91,2024-11-04 08:09:35
2,TX000003,AC00019,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,1,1122.35,2024-11-04 08:07:04
3,TX000004,AC00070,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,1,8569.06,2024-11-04 08:09:06
4,TX000005,AC00411,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40,2024-11-04 08:06:39


In [7]:
df_customer.head()

,customer_id,first_name,surname,gender,date_of_birth,age,marital,education,job,income,contact,geography,balance,has_cr_card,default,housing,loan,credit_score,tenure,dt_customer
0,C2540,Patrick,Hargrave,Female,1983-03-17,42,SINGLE,MASTER,self-employed,66726.0,LANDLINE,France,0.00,1,NO,NO,NO,619,11,2014-01-12
1,C1455,Jonathan,Hill,Female,1984-02-10,41,DIVORCED,UNIVERSITY,technician,41883.0,MOBILE,Spain,83807.86,0,UNKNOWN,YES,NO,608,12,2013-03-19
2,C2701,Jennifer,Onio,Female,1983-01-28,42,SINGLE,UNIVERSITY,student,43824.0,MOBILE,France,159660.80,1,NO,NO,YES,502,13,2012-09-15
3,C5074,James,Boni,Female,1986-10-11,39,MARRIED,PHD,admin.,65640.0,MOBILE,France,0.00,0,UNKNOWN,YES,NO,699,11,2014-03-01
4,C7320,Heather,Mitchell,Female,1982-06-03,43,SINGLE,UNIVERSITY,technician,48686.0,LANDLINE,Spain,125510.82,1,NO,YES,NO,850,12,2013-12-04


In [8]:
print("Customer shape:", df_customer.shape)
print("Bank trx shape:", df_bank_trx.shape)

Customer shape: (6121, 20)
Bank trx shape: (2512, 16)


In [9]:
print("Customer columns:", df_customer.columns)
print("Bank trx columns:", df_bank_trx.columns)

Customer columns: Index(['customer_id', 'first_name', 'surname', 'gender', 'date_of_birth',
       'age', 'marital', 'education', 'job', 'income', 'contact', 'geography',
       'balance', 'has_cr_card', 'default', 'housing', 'loan', 'credit_score',
       'tenure', 'dt_customer'],
      dtype='object')
Bank trx columns: Index(['TransactionID', 'AccountID', 'TransactionAmount', 'TransactionDate',
       'TransactionType', 'Location', 'DeviceID', 'IP Address', 'MerchantID',
       'Channel', 'CustomerAge', 'CustomerOccupation', 'TransactionDuration',
       'LoginAttempts', 'AccountBalance', 'PreviousTransactionDate'],
      dtype='object')


In [10]:
duplicate_data = df_bank_trx.duplicated(keep=False)

print(duplicate_data)

0       False
1       False
2       False
3       False
4       False
        ...  
2507    False
2508    False
2509    False
2510    False
2511    False
Length: 2512, dtype: bool


In [11]:
duplicate_data_customer = df_customer.duplicated(keep=False)

print(duplicate_data_customer)

0       False
1       False
2       False
3       False
4       False
        ...  
6116    False
6117    False
6118    False
6119    False
6120    False
Length: 6121, dtype: bool


## **Standardization & Normalization**

For the transaction data, the following standardization and normalization steps are applied to ensure a consistent and usable schema:

- Clean and unify column names
    All column names are converted to lowercase, and selected fields from the raw dataset are renamed (e.g., Location → location_device).

- Parse and standardize datetime fields
    transaction_date is parsed into a proper datetime format, and later used for ordering and deriving previous_transaction_date.

- Normalize channel values
    Raw channel strings are cleaned and mapped into consistent labels such as: ATM, WEB, MOBILE, POS, and BRANCH.

- Enforce numeric type consistency
    Transaction amounts, durations, and login attempts are cast into numeric formats.

- Standardize device and network attributes
    Fields such as device_id, ip_address, and location_device are normalized to a uniform structure.

This process ensures that the raw transaction input is transformed into a clean, structured dataset ready for feature engineering, customer enrichment, and synthetic routing generation.

In [12]:
# standardize the columns name

#------------------------------------------
#df_bank_trx.columns = (
#    df_bank_trx.columns
#        .str.strip()
#        .str.lower()
#        .str.replace(" ", "_")
#        .str.replace("-", "_")
#)


#df_bank_trx = df_bank_trx.rename(columns={
#    "transactionid": "transaction_id",
#    "accountid": "account_id",
#    "transactionamount": "transaction_amount",
#    "transactiondate": "transaction_date",
#    "transactiontype": "transaction_type",
#    "merchantid": "merchant_id",
#    "deviceid": "device_id",
#    "ip_address": "ip_address",
#    "merchantid": "merchant_id",
#    "customerage": "customer_age",
#    "customeroccupation": "customer_occupation",
#    "transactionduration": "transaction_duration",
#    "loginattempts": "login_attempts",
#    "accountbalance": "account_balance",
#    "previoustransactiondate": "previous_transaction_date"
#})

#print("Bank trx columns:", df_bank_trx.columns)

#-------------------------------------------------------------


df_trx = df_bank_trx.rename(columns={
    "TransactionID": "transaction_id",
    "AccountID": "account_id_source",        # simpan ID akun asal (AC00xxx)
    "TransactionAmount": "transaction_amount",
    "TransactionDate": "transaction_date",
    "TransactionType": "transaction_type",
    "Location": "location_device",
    "DeviceID": "device_id",
    "IP Address": "ip_address",
    "MerchantID": "merchant_id",
    "Channel": "channel",
    "CustomerAge": "customer_age",
    "CustomerOccupation": "customer_occupation",
    "TransactionDuration": "transaction_duration",
    "LoginAttempts": "login_attempts",
    "AccountBalance": "account_balance",
    "PreviousTransactionDate": "previous_transaction_date_raw"  # kalau mau, nanti bisa di-drop
})


print("Bank trx columns:", df_bank_trx.columns)

Bank trx columns: Index(['TransactionID', 'AccountID', 'TransactionAmount', 'TransactionDate',
       'TransactionType', 'Location', 'DeviceID', 'IP Address', 'MerchantID',
       'Channel', 'CustomerAge', 'CustomerOccupation', 'TransactionDuration',
       'LoginAttempts', 'AccountBalance', 'PreviousTransactionDate'],
      dtype='object')


In [13]:
df_trx.head()

,transaction_id,account_id_source,transaction_amount,transaction_date,transaction_type,location_device,device_id,ip_address,merchant_id,channel,customer_age,customer_occupation,transaction_duration,login_attempts,account_balance,previous_transaction_date_raw
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21,2024-11-04 08:08:08
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91,2024-11-04 08:09:35
2,TX000003,AC00019,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,1,1122.35,2024-11-04 08:07:04
3,TX000004,AC00070,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,1,8569.06,2024-11-04 08:09:06
4,TX000005,AC00411,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40,2024-11-04 08:06:39


In [14]:
df_trx.shape

(2512, 16)

In [15]:
df_cust = df_customer.copy()

In [16]:
# Standardize date

df_trx["transaction_date"] = pd.to_datetime(df_trx["transaction_date"], errors="coerce")
df_cust["dt_customer"] = pd.to_datetime(df_cust["dt_customer"], errors="coerce")
df_cust["date_of_birth"] = pd.to_datetime(df_cust["date_of_birth"], errors="coerce")


In [17]:
df_trx.head()

,transaction_id,account_id_source,transaction_amount,transaction_date,transaction_type,location_device,device_id,ip_address,merchant_id,channel,customer_age,customer_occupation,transaction_duration,login_attempts,account_balance,previous_transaction_date_raw
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21,2024-11-04 08:08:08
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91,2024-11-04 08:09:35
2,TX000003,AC00019,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,1,1122.35,2024-11-04 08:07:04
3,TX000004,AC00070,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,1,8569.06,2024-11-04 08:09:06
4,TX000005,AC00411,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40,2024-11-04 08:06:39


In [18]:
#dedup customer master 

df_cust = (
    df_cust
    .drop_duplicates(subset="customer_id", keep="first")
    .reset_index(drop=True)
)

print("Unique customers:", df_cust["customer_id"].nunique())

Unique customers: 6121


In [19]:
print("Customer dataset (Length):", len(df_customer))
print("Transaction dataset (Length):", len(df_trx))

Customer dataset (Length): 6121
Transaction dataset (Length): 2512


**Assumption:**

In the real banking data, customer not always have transactions, their transaction maybe dormant or they only register themselves.

1 customer_id can have many account_id, and 1 account_id can have many transaction_id

In [20]:
# get the unique account_id from the transaction dataset

account_unique = df_trx["account_id_source"].unique()
print("Unique accounts in transactions:", len(account_unique))

Unique accounts in transactions: 495


In [21]:
# get the customer data as many as the unique account

unique_accounts = df_trx["account_id_source"].unique()
n_accounts = len(unique_accounts)

if len(df_customer) < n_accounts:
    raise ValueError("the number of account is smaller than number of account!")

selected_customer = df_customer.sample(n=n_accounts, replace=False)

selected_customer.head()

,customer_id,first_name,surname,gender,date_of_birth,age,marital,education,job,income,contact,geography,balance,has_cr_card,default,housing,loan,credit_score,tenure,dt_customer
362,C4036,Tammy,Nolan,Female,1983-11-25,42,MARRIED,UNIVERSITY,blue-collar,44964.0,MOBILE,Germany,87271.41,1,NO,NO,YES,540,13,2012-12-16
2739,C9421,Andrea,Yao,Male,1990-06-15,35,SINGLE,UNIVERSITY,services,48070.0,LANDLINE,Germany,105346.03,1,NO,YES,NO,603,12,2013-01-13
2539,C9156,Charles,Bazhenov,Male,2000-11-15,25,MARRIED,UNIVERSITY,admin.,62503.0,LANDLINE,France,162560.32,1,NO,YES,YES,530,12,2013-02-18
5493,C3835,Sarah,Tochukwu,Female,2000-02-19,25,MARRIED,PROFESSIONAL,services,32632.0,LANDLINE,Germany,128524.19,1,NO,YES,YES,553,13,2012-08-02
3606,C8580,Christopher,Chidimma,Female,1973-06-18,52,MARRIED,UNIVERSITY,blue-collar,35196.0,MOBILE,Germany,86891.84,1,NO,YES,NO,660,13,2012-11-13


In [22]:
df_customer.columns

Index(['customer_id', 'first_name', 'surname', 'gender', 'date_of_birth',
       'age', 'marital', 'education', 'job', 'income', 'contact', 'geography',
       'balance', 'has_cr_card', 'default', 'housing', 'loan', 'credit_score',
       'tenure', 'dt_customer'],
      dtype='object')

In [23]:
df_cust = selected_customer[['customer_id', 'first_name', 'surname','balance','dt_customer']]

df_cust.head()

,customer_id,first_name,surname,balance,dt_customer
362,C4036,Tammy,Nolan,87271.41,2012-12-16
2739,C9421,Andrea,Yao,105346.03,2013-01-13
2539,C9156,Charles,Bazhenov,162560.32,2013-02-18
5493,C3835,Sarah,Tochukwu,128524.19,2012-08-02
3606,C8580,Christopher,Chidimma,86891.84,2012-11-13


## **Create Account Master**

In [24]:
account_master = pd.DataFrame({
    "account_id": unique_accounts,
    "customer_id": df_cust["customer_id"].values
})

In [25]:
# optional fields
account_master["account_type"] = np.random.choice(
    ["CURRENT", "SAVINGS", "BUSINESS"], size=n_accounts
)
account_master["currency"] = np.random.choice(["EUR", "USD"], size=n_accounts)


In [26]:
account_master.head()

,account_id,customer_id,account_type,currency
0,AC00128,C4036,CURRENT,EUR
1,AC00455,C9421,BUSINESS,EUR
2,AC00019,C9156,SAVINGS,EUR
3,AC00070,C3835,CURRENT,USD
4,AC00411,C8580,SAVINGS,EUR


In [27]:
# Generate account_id (IBAN-like) 

from faker import Faker
import random
fake = Faker()

def generate_estonian_iban():
    return "EE" + str(random.randint(10, 99)) + str(random.randint(10**15, 10**16 - 1))


# generate iban to every account in the df_account
account_master["account_id_new"] = [generate_estonian_iban() for _ in range(len(account_master))]



In [28]:
account_master.head()

,account_id,customer_id,account_type,currency,account_id_new
0,AC00128,C4036,CURRENT,EUR,EE813389480738021154
1,AC00455,C9421,BUSINESS,EUR,EE753135455302083983
2,AC00019,C9156,SAVINGS,EUR,EE807731860600978103
3,AC00070,C3835,CURRENT,USD,EE551023314553680719
4,AC00411,C8580,SAVINGS,EUR,EE316167583968429179


In [29]:
account_master = account_master.rename(columns={
    "account_id": "account_id_old",
    "account_id_new": "account_id"
})

In [30]:
df_trx.head()

,transaction_id,account_id_source,transaction_amount,transaction_date,transaction_type,location_device,device_id,ip_address,merchant_id,channel,customer_age,customer_occupation,transaction_duration,login_attempts,account_balance,previous_transaction_date_raw
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21,2024-11-04 08:08:08
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91,2024-11-04 08:09:35
2,TX000003,AC00019,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,1,1122.35,2024-11-04 08:07:04
3,TX000004,AC00070,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,1,8569.06,2024-11-04 08:09:06
4,TX000005,AC00411,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40,2024-11-04 08:06:39


In [31]:
#df_trx = df_trx.merge(
#    df_account[["account_id_old", "account_id", "customer_id"]],
#    left_on="account_id",
#    right_on="account_id_old",
#    how="left",
#    validate="many_to_one"
#)


df_transactions_final = df_trx.merge(
    account_master,
    left_on="account_id_source",
    right_on="account_id_old",
    how="left"
)

In [32]:
df_transactions_final.head()

,transaction_id,account_id_source,transaction_amount,transaction_date,transaction_type,location_device,device_id,ip_address,merchant_id,channel,...,customer_occupation,transaction_duration,login_attempts,account_balance,previous_transaction_date_raw,account_id_old,customer_id,account_type,currency,account_id
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,...,Doctor,81,1,5112.21,2024-11-04 08:08:08,AC00128,C4036,CURRENT,EUR,EE813389480738021154
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,...,Doctor,141,1,13758.91,2024-11-04 08:09:35,AC00455,C9421,BUSINESS,EUR,EE753135455302083983
2,TX000003,AC00019,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,...,Student,56,1,1122.35,2024-11-04 08:07:04,AC00019,C9156,SAVINGS,EUR,EE807731860600978103
3,TX000004,AC00070,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,...,Student,25,1,8569.06,2024-11-04 08:09:06,AC00070,C3835,CURRENT,USD,EE551023314553680719
4,TX000005,AC00411,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,...,Student,198,1,7429.40,2024-11-04 08:06:39,AC00411,C8580,SAVINGS,EUR,EE316167583968429179


In [33]:
# drop old account_id
df_transactions_final = df_transactions_final.drop(columns=["account_id_source", "account_id_old"])



In [34]:
df_transactions_final.head()

,transaction_id,transaction_amount,transaction_date,transaction_type,location_device,device_id,ip_address,merchant_id,channel,customer_age,customer_occupation,transaction_duration,login_attempts,account_balance,previous_transaction_date_raw,customer_id,account_type,currency,account_id
0,TX000001,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21,2024-11-04 08:08:08,C4036,CURRENT,EUR,EE813389480738021154
1,TX000002,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91,2024-11-04 08:09:35,C9421,BUSINESS,EUR,EE753135455302083983
2,TX000003,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,1,1122.35,2024-11-04 08:07:04,C9156,SAVINGS,EUR,EE807731860600978103
3,TX000004,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,1,8569.06,2024-11-04 08:09:06,C3835,CURRENT,USD,EE551023314553680719
4,TX000005,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40,2024-11-04 08:06:39,C8580,SAVINGS,EUR,EE316167583968429179


In [35]:
s = df_transactions_final.duplicated(keep=False)

print(s)

0       False
1       False
2       False
3       False
4       False
        ...  
2507    False
2508    False
2509    False
2510    False
2511    False
Length: 2512, dtype: bool


In [36]:
df_cust.columns

Index(['customer_id', 'first_name', 'surname', 'balance', 'dt_customer'], dtype='object')

In [37]:
duplicate = df_cust["customer_id"].nunique(), len(df_customer)
print(duplicate)

(495, 6121)


In [38]:
df_transactions_final = df_transactions_final.merge(
    df_cust,
    on="customer_id",
    how="left",
    validate="many_to_one"
)


In [39]:
check_data_validity = df_transactions_final["transaction_date"] < df_transactions_final["dt_customer"]
df_transactions_final.loc[check_data_validity, "transaction_date"] = df_transactions_final.loc[check_data_validity, "dt_customer"]

In [40]:
df_transactions_final.head()

,transaction_id,transaction_amount,transaction_date,transaction_type,location_device,device_id,ip_address,merchant_id,channel,customer_age,...,account_balance,previous_transaction_date_raw,customer_id,account_type,currency,account_id,first_name,surname,balance,dt_customer
0,TX000001,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,...,5112.21,2024-11-04 08:08:08,C4036,CURRENT,EUR,EE813389480738021154,Tammy,Nolan,87271.41,2012-12-16
1,TX000002,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,...,13758.91,2024-11-04 08:09:35,C9421,BUSINESS,EUR,EE753135455302083983,Andrea,Yao,105346.03,2013-01-13
2,TX000003,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,...,1122.35,2024-11-04 08:07:04,C9156,SAVINGS,EUR,EE807731860600978103,Charles,Bazhenov,162560.32,2013-02-18
3,TX000004,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,...,8569.06,2024-11-04 08:09:06,C3835,CURRENT,USD,EE551023314553680719,Sarah,Tochukwu,128524.19,2012-08-02
4,TX000005,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,...,7429.40,2024-11-04 08:06:39,C8580,SAVINGS,EUR,EE316167583968429179,Christopher,Chidimma,86891.84,2012-11-13


### **Asumption**

| previous_transaction_date | dt_customer | max            | Explanation                                     |
| ------------------------- | ----------- | -------------- | ----------------------------------------------- |
| 2020-01-01                | 2022-05-10  | **2022-05-10** |invalid, so if previous_transaction_date < dt_customer pick the max date |
| NaT                       | 2022-05-10  | **2022-05-10** | invalid. if Null input with dt_customer |
| 2023-03-01                | 2022-05-10  | **2023-03-01** | valid, previous_transaction_date > dt_customer pick the max date|


In [41]:
# Previous Transaction Date

#1. Short transaction based on date (oldest to newest date)
df_transactions_final = df_transactions_final.sort_values(["customer_id", "transaction_date"])

#2. take the previous transaction date for each customer and shift the data to the previous row
df_transactions_final["previous_transaction_date"] = df_transactions_final.groupby("customer_id")["transaction_date"].shift()

#3. Convert both date to datetime so that max() can run
df_transactions_final["previous_transaction_date"] = pd.to_datetime(df_transactions_final["previous_transaction_date"], errors="coerce")
df_transactions_final["dt_customer"] = pd.to_datetime(df_transactions_final["dt_customer"], errors="coerce")

# compare previous transaction date to dt_customer, and pick the max date 
df_transactions_final["previous_transaction_date"] = df_transactions_final[["previous_transaction_date", "dt_customer"]].max(axis=1)


In [42]:
df_transactions_final.head()

,transaction_id,transaction_amount,transaction_date,transaction_type,location_device,device_id,ip_address,merchant_id,channel,customer_age,...,previous_transaction_date_raw,customer_id,account_type,currency,account_id,first_name,surname,balance,dt_customer,previous_transaction_date
1909,TX001910,301.47,2023-01-26 16:13:15,Debit,Colorado Springs,D000030,36.13.239.172,M040,Branch,21,...,2024-11-04 08:09:25,C1006,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2013-08-21 00:00:00
1241,TX001242,168.11,2023-05-19 16:44:53,Credit,New York,D000050,146.69.70.214,M015,Online,51,...,2024-11-04 08:06:59,C1006,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2023-01-26 16:13:15
1054,TX001055,382.07,2023-06-05 17:48:07,Credit,Omaha,D000318,21.97.154.92,M091,Branch,62,...,2024-11-04 08:08:17,C1006,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2023-05-19 16:44:53
1746,TX001747,305.18,2023-06-05 17:48:33,Debit,Chicago,D000699,93.146.251.20,M065,Online,58,...,2024-11-04 08:08:22,C1006,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2023-06-05 17:48:07
1189,TX001190,7.95,2023-01-09 16:45:12,Debit,Phoenix,D000147,101.120.142.50,M087,ATM,79,...,2024-11-04 08:10:06,C1059,BUSINESS,EUR,EE439018519889093488,Johnny,Wilkins,0.0,2013-06-29,2013-06-29 00:00:00


## **Generate Account Created Date**

In [43]:
df = df_transactions_final.copy()

In [44]:
df.groupby("customer_id")["account_id"].nunique().value_counts()

account_id
1    495
Name: count, dtype: int64

In [45]:
# check for multi account possibility

multi_accounts = df.groupby("customer_id")["account_id"].nunique().reset_index().query("account_id > 1").head(10)

In [46]:
df["account_created_date"] = df["dt_customer"]

df.head()

,transaction_id,transaction_amount,transaction_date,transaction_type,location_device,device_id,ip_address,merchant_id,channel,customer_age,...,customer_id,account_type,currency,account_id,first_name,surname,balance,dt_customer,previous_transaction_date,account_created_date
1909,TX001910,301.47,2023-01-26 16:13:15,Debit,Colorado Springs,D000030,36.13.239.172,M040,Branch,21,...,C1006,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2013-08-21 00:00:00,2013-08-21
1241,TX001242,168.11,2023-05-19 16:44:53,Credit,New York,D000050,146.69.70.214,M015,Online,51,...,C1006,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2023-01-26 16:13:15,2013-08-21
1054,TX001055,382.07,2023-06-05 17:48:07,Credit,Omaha,D000318,21.97.154.92,M091,Branch,62,...,C1006,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2023-05-19 16:44:53,2013-08-21
1746,TX001747,305.18,2023-06-05 17:48:33,Debit,Chicago,D000699,93.146.251.20,M065,Online,58,...,C1006,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2023-06-05 17:48:07,2013-08-21
1189,TX001190,7.95,2023-01-09 16:45:12,Debit,Phoenix,D000147,101.120.142.50,M087,ATM,79,...,C1059,BUSINESS,EUR,EE439018519889093488,Johnny,Wilkins,0.0,2013-06-29,2013-06-29 00:00:00,2013-06-29


In [47]:
# mapping remapping data location

# 1. Create region & cities dictionary

cities = {
    "Estonia": [
        "Tallinn", "Tartu", "Narva", "Pärnu", "Viljandi",
        "Kuressaare", "Rakvere", "Sillamäe"
    ],
    "Europe": [
        "Helsinki", "Stockholm", "Oslo", "Copenhagen", "Berlin",
        "Paris", "Madrid", "Rome", "Warsaw", "Prague"
    ],
    "Asia": [
        "Jakarta", "Singapore", "Tokyo", "Seoul", "Bangkok",
        "Kuala Lumpur", "Dubai", "Doha", "Sharjah", "Jeddah", "Dammam", "Yangon"
    ],
    "America": [
        "New York", "Los Angeles", "San Francisco", "Chicago",
        "Toronto", "Vancouver", "Mexico City", "San Diego", "Houston"
    ],
    "Oceania": [
        "Sydney", "Melbourne" 
    ],
    "Africa": [
        "Cape Town", "Johannesburg", "Lagos","Nairobi"
    ]
}

# 2. Assign Region With Priority for Estonia

region_weights = {
    "Estonia": 0.70,
    "Europe": 0.20,
    "Asia": 0.05,
    "America": 0.05
}

# 3. Assign Region based on weight

def pick_region():
    return random.choices(
        population=list(region_weights.keys()),
        weights=list(region_weights.values()),
        k=1
    )[0]


# 4. Assign a City Based on the Region

def pick_city(region):
    return random.choice(cities[region])

# Return a Random City With Estonia Priority

def random_city():
    region = pick_region()
    return pick_city(region)

df.loc[:, "location"] = df.apply(
    lambda x: random_city(), axis=1
)


In [48]:
df.head()

,transaction_id,transaction_amount,transaction_date,transaction_type,location_device,device_id,ip_address,merchant_id,channel,customer_age,...,account_type,currency,account_id,first_name,surname,balance,dt_customer,previous_transaction_date,account_created_date,location
1909,TX001910,301.47,2023-01-26 16:13:15,Debit,Colorado Springs,D000030,36.13.239.172,M040,Branch,21,...,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2013-08-21 00:00:00,2013-08-21,Jakarta
1241,TX001242,168.11,2023-05-19 16:44:53,Credit,New York,D000050,146.69.70.214,M015,Online,51,...,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2023-01-26 16:13:15,2013-08-21,Kuressaare
1054,TX001055,382.07,2023-06-05 17:48:07,Credit,Omaha,D000318,21.97.154.92,M091,Branch,62,...,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2023-05-19 16:44:53,2013-08-21,Rakvere
1746,TX001747,305.18,2023-06-05 17:48:33,Debit,Chicago,D000699,93.146.251.20,M065,Online,58,...,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2023-06-05 17:48:07,2013-08-21,Yangon
1189,TX001190,7.95,2023-01-09 16:45:12,Debit,Phoenix,D000147,101.120.142.50,M087,ATM,79,...,BUSINESS,EUR,EE439018519889093488,Johnny,Wilkins,0.0,2013-06-29,2013-06-29 00:00:00,2013-06-29,Narva


In [49]:
# check for mismatch data

df_check = df.merge(
    df_cust[["customer_id", "first_name", "surname"]],
    on="customer_id",
    how="left",
    suffixes=("_txn", "_cust")
)

df_check[df_check["first_name_txn"] != df_check["first_name_cust"]]


,transaction_id,transaction_amount,transaction_date,transaction_type,location_device,device_id,ip_address,merchant_id,channel,customer_age,...,account_id,first_name_txn,surname_txn,balance,dt_customer,previous_transaction_date,account_created_date,location,first_name_cust,surname_cust


In [50]:
# check data completeness
df.iloc[1].T

transaction_id                               TX001242
transaction_amount                             168.11
transaction_date                  2023-05-19 16:44:53
transaction_type                               Credit
location_device                              New York
device_id                                     D000050
ip_address                              146.69.70.214
merchant_id                                      M015
channel                                        Online
customer_age                                       51
customer_occupation                            Doctor
transaction_duration                               12
login_attempts                                      1
account_balance                              10652.13
previous_transaction_date_raw     2024-11-04 08:06:59
customer_id                                     C1006
account_type                                  SAVINGS
currency                                          USD
account_id                  

In [51]:
df.head()

,transaction_id,transaction_amount,transaction_date,transaction_type,location_device,device_id,ip_address,merchant_id,channel,customer_age,...,account_type,currency,account_id,first_name,surname,balance,dt_customer,previous_transaction_date,account_created_date,location
1909,TX001910,301.47,2023-01-26 16:13:15,Debit,Colorado Springs,D000030,36.13.239.172,M040,Branch,21,...,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2013-08-21 00:00:00,2013-08-21,Jakarta
1241,TX001242,168.11,2023-05-19 16:44:53,Credit,New York,D000050,146.69.70.214,M015,Online,51,...,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2023-01-26 16:13:15,2013-08-21,Kuressaare
1054,TX001055,382.07,2023-06-05 17:48:07,Credit,Omaha,D000318,21.97.154.92,M091,Branch,62,...,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2023-05-19 16:44:53,2013-08-21,Rakvere
1746,TX001747,305.18,2023-06-05 17:48:33,Debit,Chicago,D000699,93.146.251.20,M065,Online,58,...,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2023-06-05 17:48:07,2013-08-21,Yangon
1189,TX001190,7.95,2023-01-09 16:45:12,Debit,Phoenix,D000147,101.120.142.50,M087,ATM,79,...,BUSINESS,EUR,EE439018519889093488,Johnny,Wilkins,0.0,2013-06-29,2013-06-29 00:00:00,2013-06-29,Narva


In [52]:
df.shape

(2512, 26)

In [53]:
# Clean numeric fields (amount, duration, login attempts) to standardize the data

df["transaction_amount"] = pd.to_numeric(df["transaction_amount"], errors="coerce")
df["transaction_duration"] = pd.to_numeric(df["transaction_duration"], errors="coerce")
df["login_attempts"] = pd.to_numeric(df["login_attempts"], errors="coerce")

In [54]:
#standardixe channel values

def clean_channel(x):
    if pd.isna(x):
        return "UNKNOWN"

    x = str(x).strip().upper()

    if "ATM" in x:
        return "ATM"
    if any(k in x for k in ["WEB", "ONLINE", "INTERNET"]):
        return "INTERNET BANKING"
    if any(k in x for k in ["MOBILE", "APP"]):
        return "MOBILE BANKING"
    if any(k in x for k in ["POS", "EDC", "CARD"]):
        return "EDC"
    if "BRANCH" in x:
        return "BRANCH"

    return x


# apply dataset df_trx

df["channel"] = df["channel"].apply(clean_channel)

df.head()

,transaction_id,transaction_amount,transaction_date,transaction_type,location_device,device_id,ip_address,merchant_id,channel,customer_age,...,account_type,currency,account_id,first_name,surname,balance,dt_customer,previous_transaction_date,account_created_date,location
1909,TX001910,301.47,2023-01-26 16:13:15,Debit,Colorado Springs,D000030,36.13.239.172,M040,BRANCH,21,...,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2013-08-21 00:00:00,2013-08-21,Jakarta
1241,TX001242,168.11,2023-05-19 16:44:53,Credit,New York,D000050,146.69.70.214,M015,INTERNET BANKING,51,...,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2023-01-26 16:13:15,2013-08-21,Kuressaare
1054,TX001055,382.07,2023-06-05 17:48:07,Credit,Omaha,D000318,21.97.154.92,M091,BRANCH,62,...,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2023-05-19 16:44:53,2013-08-21,Rakvere
1746,TX001747,305.18,2023-06-05 17:48:33,Debit,Chicago,D000699,93.146.251.20,M065,INTERNET BANKING,58,...,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2023-06-05 17:48:07,2013-08-21,Yangon
1189,TX001190,7.95,2023-01-09 16:45:12,Debit,Phoenix,D000147,101.120.142.50,M087,ATM,79,...,BUSINESS,EUR,EE439018519889093488,Johnny,Wilkins,0.0,2013-06-29,2013-06-29 00:00:00,2013-06-29,Narva


In [55]:
df.iloc[1].T

transaction_id                               TX001242
transaction_amount                             168.11
transaction_date                  2023-05-19 16:44:53
transaction_type                               Credit
location_device                              New York
device_id                                     D000050
ip_address                              146.69.70.214
merchant_id                                      M015
channel                              INTERNET BANKING
customer_age                                       51
customer_occupation                            Doctor
transaction_duration                               12
login_attempts                                      1
account_balance                              10652.13
previous_transaction_date_raw     2024-11-04 08:06:59
customer_id                                     C1006
account_type                                  SAVINGS
currency                                          USD
account_id                  

In [56]:
df['transaction_type'].unique()

array(['Debit', 'Credit'], dtype=object)

In [57]:
df.head()

,transaction_id,transaction_amount,transaction_date,transaction_type,location_device,device_id,ip_address,merchant_id,channel,customer_age,...,account_type,currency,account_id,first_name,surname,balance,dt_customer,previous_transaction_date,account_created_date,location
1909,TX001910,301.47,2023-01-26 16:13:15,Debit,Colorado Springs,D000030,36.13.239.172,M040,BRANCH,21,...,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2013-08-21 00:00:00,2013-08-21,Jakarta
1241,TX001242,168.11,2023-05-19 16:44:53,Credit,New York,D000050,146.69.70.214,M015,INTERNET BANKING,51,...,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2023-01-26 16:13:15,2013-08-21,Kuressaare
1054,TX001055,382.07,2023-06-05 17:48:07,Credit,Omaha,D000318,21.97.154.92,M091,BRANCH,62,...,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2023-05-19 16:44:53,2013-08-21,Rakvere
1746,TX001747,305.18,2023-06-05 17:48:33,Debit,Chicago,D000699,93.146.251.20,M065,INTERNET BANKING,58,...,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2023-06-05 17:48:07,2013-08-21,Yangon
1189,TX001190,7.95,2023-01-09 16:45:12,Debit,Phoenix,D000147,101.120.142.50,M087,ATM,79,...,BUSINESS,EUR,EE439018519889093488,Johnny,Wilkins,0.0,2013-06-29,2013-06-29 00:00:00,2013-06-29,Narva


In [58]:
df = df.reset_index(drop=True)

In [59]:
df.head()

,transaction_id,transaction_amount,transaction_date,transaction_type,location_device,device_id,ip_address,merchant_id,channel,customer_age,...,account_type,currency,account_id,first_name,surname,balance,dt_customer,previous_transaction_date,account_created_date,location
0,TX001910,301.47,2023-01-26 16:13:15,Debit,Colorado Springs,D000030,36.13.239.172,M040,BRANCH,21,...,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2013-08-21 00:00:00,2013-08-21,Jakarta
1,TX001242,168.11,2023-05-19 16:44:53,Credit,New York,D000050,146.69.70.214,M015,INTERNET BANKING,51,...,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2023-01-26 16:13:15,2013-08-21,Kuressaare
2,TX001055,382.07,2023-06-05 17:48:07,Credit,Omaha,D000318,21.97.154.92,M091,BRANCH,62,...,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2023-05-19 16:44:53,2013-08-21,Rakvere
3,TX001747,305.18,2023-06-05 17:48:33,Debit,Chicago,D000699,93.146.251.20,M065,INTERNET BANKING,58,...,SAVINGS,USD,EE816402671097545968,Hannah,Napolitani,0.0,2013-08-21,2023-06-05 17:48:07,2013-08-21,Yangon
4,TX001190,7.95,2023-01-09 16:45:12,Debit,Phoenix,D000147,101.120.142.50,M087,ATM,79,...,BUSINESS,EUR,EE439018519889093488,Johnny,Wilkins,0.0,2013-06-29,2013-06-29 00:00:00,2013-06-29,Narva


## **Adjustment for taking like Paysim Data**

| Payment Instrument / Region | Fraud Rate (By Value) | Fraud Rate (By Volume) | Key Insights | Source |
|-----------------------------|------------------------|--------------------------|--------------|--------|
| **Card Payments (EEA, H1 2023)** | **0.031%** | **0.015%** | Highest fraud rate among payment instruments; still extremely low overall. | EBA–ECB Joint Report on Payment Fraud (2024) |
| **E-Money Payments (EEA, H1 2023)** | **0.022%** | **0.012%** | Similar low levels to card fraud; often affected by low-value scams. | EBA–ECB Joint Report on Payment Fraud (2024) |
| **Credit Transfers (EEA)** | <0.005% | <0.005% | Very low fraud rates due to SCA and strong monitoring controls. | EBA–ECB Joint Report on Payment Fraud (2024) |
| **Direct Debits (EEA)** | ~0.001% | ~0.001% | Lowest fraud rate among all payment types. | EBA–ECB Joint Report on Payment Fraud (2024) |
| **Card-Not-Present (CNP) Fraud Reduction After SCA** | ↓ **77% (value)** | ↓ **55% (volume)** | SCA significantly reduces fraudulent online payment attempts. | EBA Opinion on New Types of Payment Fraud (2024) |
| **UK Purchase Fraud (2024)** | n/a | n/a | **94% of scams are low-value (<£1,000)**; overall fraud losses decreased to £1.17bn. | UK Finance Annual Fraud Report (2024) |
| **ML/TF High-Risk Payments (EU)** | Very low prevalence | n/a | Rare but high-impact events; still <<0.1% of transactions. | EBA Opinion on ML/TF Risks (2025) |


- Fraud rates in real banking systems are extremely low (<0.05%).

- Fraud does not dominate transaction datasets.

- High-value fraud is rare; most scams are low-value and low-frequency.


In this data pipeline, I will inject only 0.5% of transaction with paysim, just to show the realistic fraud transaction based on the overall banking report globally. 


```mermaid
flowchart LR
    A["Bank Transactions<br>(Baseline Real Data)<br>~100% normal"] --> C[Merge]
    B["PaySim AML Data<br>(High fraud density)"] --> D["Sample Only 1–10%<br>(Maintain Realism)"]
    D --> C

    C --> E["TRANSACTION_MASTER<br>Mostly real transactions<br>+ small AML injection"]

    style A fill:#e6f7ff,stroke:#33a1ff,stroke-width:2px
    style B fill:#ffe6e6,stroke:#ff4d4d,stroke-width:2px
    style D fill:#fff3cd,stroke:#ffcc00,stroke-width:2px
    style C fill:#e8ffe6,stroke:#00cc66,stroke-width:2px
    style E fill:#d4fcd9,stroke:#00994d,stroke-width:2px

```

the name_orig & name_dect will be change as account ID sender and receiver and itshould be align with transaction account ID

In [60]:
# Mapping city ti COuntry (based on previous list 'location')


city_to_country = {
    # ----- Estonia -----
    "Tallinn": "Estonia",
    "Tartu": "Estonia",
    "Narva": "Estonia",
    "Pärnu": "Estonia",
    "Viljandi": "Estonia",
    "Kuressaare": "Estonia",
    "Rakvere": "Estonia",
    "Sillamäe": "Estonia",
    
    # ----- Europe -----
    "Helsinki": "Finland",
    "Stockholm": "Sweden",
    "Oslo": "Norway",
    "Copenhagen": "Denmark",
    "Berlin": "Germany",
    "Frankfurt": "Germany",
    "Paris": "France",
    "Madrid": "Spain",
    "Rome": "Italy",
    "Warsaw": "Poland",
    "Prague": "Czech Republic",

    # ----- Asia -----
    "Jakarta": "Indonesia",
    "Singapore": "Singapore",
    "Tokyo": "Japan",
    "Seoul": "South Korea",
    "Bangkok": "Thailand",
    "Kuala Lumpur": "Malaysia",
    "Dubai": "United Arab Emirates",
    "Doha": "Qatar",
    "Yangon": "Myanmar",
    "Dammam": "Saudi Arabia",
    "Jeddah": "Saudi Arabia",
    "Sharjah": "United Arab Emirates",
    "Qatar": "Qatar",

    # ----- America -----
    "New York": "USA",
    "Los Angeles": "USA",
    "San Francisco": "USA",
    "Chicago": "USA",
    "Toronto": "Canada",
    "Vancouver": "Canada",
    "Mexico City": "Mexico",
    "San Diego": "USA",
    "Houston": "USA",
    "Miami": "USA",
    "Raleigh": "USA",
    "Nashville": "USA",
    "Mesa": "USA",
    "Atlanta": "USA",
    "Dallas": "USA",
    "Phoenix": "USA",
    "Reno": "USA",

    # ----- Other -----
    "Sydney": "Australia",
    "Melbourne": "Australia",
    "Cape Town": "South Africa",
    "Johannesburg": "South Africa",
    "Lagos": "Nigeria",
    "Nairobi": "Kenya",
    "South Africa": "South Africa"
}


In [61]:
# Map Country to Region

country_to_region = {
    "Estonia": "Europe",
    "Finland": "Europe",
    "Sweden": "Europe",
    "Germany": "Europe",
    "Norway": "Europe",
    "Denmark": "Europe",
    "France": "Europe",
    "Spain": "Europe",
    "Italy": "Europe",
    "Poland": "Europe",
    "Czech Republic": "Europe",
    "Japan": "Asia",
    "South Korea": "Asia",
    "Indonesia": "Asia",
    "Singapore": "Asia",
    "Thailand": "Asia",
    "Malaysia": "Asia",
    "United Arab Emirate": "Asia",
    "Qatar": "Asia",
    "Saudi Arabia": "Asia",
    "Myanmar": "Asia",
    "USA": "America",
    "Canada": "America",
    "Mexico": "America",
    "Australia": "Oceania",
    "South Africa": "Africa",
    "Nigeria": "Africa",
    "Kenya": "Africa"
}





In [62]:
# map to country code

country_to_country_code = {
    "Europe": ["FI", "SE", "NO", "DK", "DE", "FR", "ES", "IT", "PL", "CZ", "EE"],
    "Asia": ["ID", "SG", "JP", "KR", "TH", "MY", "AE", "QA", "SA", "MM"],
    "America": ["US", "CA", "MX"],
    "Oceania": ["AU"],
    "Africa" : ["ZA", "NG", "KE"]
}

In [63]:
df_mix = df.copy()

## **Counterpart Account**

In [64]:
# get the dictionary of city to country

#assumption: all receiver country is in Estonia because it use EE as IBAN

df_mix["receiver_country"] = "Estonia"
df_mix["receiver_region"] = "Europe"

In [65]:
df_mix.iloc[1].T

transaction_id                               TX001242
transaction_amount                             168.11
transaction_date                  2023-05-19 16:44:53
transaction_type                               Credit
location_device                              New York
device_id                                     D000050
ip_address                              146.69.70.214
merchant_id                                      M015
channel                              INTERNET BANKING
customer_age                                       51
customer_occupation                            Doctor
transaction_duration                               12
login_attempts                                      1
account_balance                              10652.13
previous_transaction_date_raw     2024-11-04 08:06:59
customer_id                                     C1006
account_type                                  SAVINGS
currency                                          USD
account_id                  

In [66]:
# get the dictionary of city to country

#assumption: all receiver country is in Estonia because it use EE as IBAN

def map_country(city):
    return city_to_country.get(city,"Unknown")

# get the dictionary of country to region

def map_region(country):
    return country_to_region.get(country, "Unknown")

#df_mix["sender_country"] = df_mix["location"].apply(map_country) --> pick randomly based on random_iban_international()
#df_mix["sender_region"] = df_mix["sender_country"].apply(map_region)


all_countries = list(country_to_region.keys())

df_mix["sender_country"] = np.random.choice(all_countries, size=len(df_mix))
df_mix["sender_region"]  = df_mix["sender_country"].map(country_to_region)


#df_mix["sender_region"] = df_mix["sender_country"].apply(lambda x: country_to_region.get(x, "Unknown"))
#df_mix["sender_region"].unique()

In [67]:
df_mix.iloc[1].T

transaction_id                               TX001242
transaction_amount                             168.11
transaction_date                  2023-05-19 16:44:53
transaction_type                               Credit
location_device                              New York
device_id                                     D000050
ip_address                              146.69.70.214
merchant_id                                      M015
channel                              INTERNET BANKING
customer_age                                       51
customer_occupation                            Doctor
transaction_duration                               12
login_attempts                                      1
account_balance                              10652.13
previous_transaction_date_raw     2024-11-04 08:06:59
customer_id                                     C1006
account_type                                  SAVINGS
currency                                          USD
account_id                  

In [68]:
df_mix.head()

,transaction_id,transaction_amount,transaction_date,transaction_type,location_device,device_id,ip_address,merchant_id,channel,customer_age,...,surname,balance,dt_customer,previous_transaction_date,account_created_date,location,receiver_country,receiver_region,sender_country,sender_region
0,TX001910,301.47,2023-01-26 16:13:15,Debit,Colorado Springs,D000030,36.13.239.172,M040,BRANCH,21,...,Napolitani,0.0,2013-08-21,2013-08-21 00:00:00,2013-08-21,Jakarta,Estonia,Europe,Spain,Europe
1,TX001242,168.11,2023-05-19 16:44:53,Credit,New York,D000050,146.69.70.214,M015,INTERNET BANKING,51,...,Napolitani,0.0,2013-08-21,2023-01-26 16:13:15,2013-08-21,Kuressaare,Estonia,Europe,Spain,Europe
2,TX001055,382.07,2023-06-05 17:48:07,Credit,Omaha,D000318,21.97.154.92,M091,BRANCH,62,...,Napolitani,0.0,2013-08-21,2023-05-19 16:44:53,2013-08-21,Rakvere,Estonia,Europe,Qatar,Asia
3,TX001747,305.18,2023-06-05 17:48:33,Debit,Chicago,D000699,93.146.251.20,M065,INTERNET BANKING,58,...,Napolitani,0.0,2013-08-21,2023-06-05 17:48:07,2013-08-21,Yangon,Estonia,Europe,Czech Republic,Europe
4,TX001190,7.95,2023-01-09 16:45:12,Debit,Phoenix,D000147,101.120.142.50,M087,ATM,79,...,Wilkins,0.0,2013-06-29,2013-06-29 00:00:00,2013-06-29,Narva,Estonia,Europe,Singapore,Asia


In [69]:
#generate IBAN Estonia

def random_iban_estonia():
    return "EE" + "".join([str(np.random.randint(0,10)) for _ in range(18)])

# generate IBAN International

def random_iban_international():
    region = random.choice(list(country_to_country_code.keys()))
    code = random.choice(country_to_country_code[region])
    return code + "".join([str(np.random.randint(0,10)) for _ in range(20)]) #just assumption


In [70]:
df_mix.columns

Index(['transaction_id', 'transaction_amount', 'transaction_date',
       'transaction_type', 'location_device', 'device_id', 'ip_address',
       'merchant_id', 'channel', 'customer_age', 'customer_occupation',
       'transaction_duration', 'login_attempts', 'account_balance',
       'previous_transaction_date_raw', 'customer_id', 'account_type',
       'currency', 'account_id', 'first_name', 'surname', 'balance',
       'dt_customer', 'previous_transaction_date', 'account_created_date',
       'location', 'receiver_country', 'receiver_region', 'sender_country',
       'sender_region'],
      dtype='object')

In [71]:
df_mix.shape

(2512, 30)

In [72]:
aml_blacklist_codes = ["IR", "SY", "KP", "SD", "MM", "RU"]

mix_choices = np.random.choice(
    ["INTERNAL", "DOMESTIC", "INTERNATIONAL"],
    size=len(df_mix),
    p=[0.65, 0.25, 0.10]  # you may adjust
)

sender_list = []

internal_accounts = df_mix["account_id"].unique().tolist()

for i, choice in enumerate(mix_choices):

    receiver_acc = df_mix.loc[i, "account_id"]

    # INTERNAL (to another ABC customer)
    if choice == "INTERNAL":
        valid_internal = [a for a in internal_accounts if a != receiver_acc]
        sender_list.append(
            random.choice(valid_internal) if valid_internal else random_iban_estonia()
        )

    # DOMESTIC (Estonia - external bank)
    elif choice == "DOMESTIC":
        sender_list.append(random_iban_estonia())

    # INTERNATIONAL (safe + AML blacklist)
    else:

        # 0.05% AML blacklist routing
        if np.random.rand() < 0.0005:
            code = random.choice(aml_blacklist_codes)
            iban = code + "".join([str(np.random.randint(0,10)) for _ in range(20)])
            sender_list.append(iban)
        else:
            sender_list.append(random_iban_international())

# assign
df_mix["sender_account"] = sender_list


In [73]:
df_mix[df_mix["account_id"].isna()]

,transaction_id,transaction_amount,transaction_date,transaction_type,location_device,device_id,ip_address,merchant_id,channel,customer_age,...,balance,dt_customer,previous_transaction_date,account_created_date,location,receiver_country,receiver_region,sender_country,sender_region,sender_account


In [74]:
df_mix.head()

,transaction_id,transaction_amount,transaction_date,transaction_type,location_device,device_id,ip_address,merchant_id,channel,customer_age,...,balance,dt_customer,previous_transaction_date,account_created_date,location,receiver_country,receiver_region,sender_country,sender_region,sender_account
0,TX001910,301.47,2023-01-26 16:13:15,Debit,Colorado Springs,D000030,36.13.239.172,M040,BRANCH,21,...,0.0,2013-08-21,2013-08-21 00:00:00,2013-08-21,Jakarta,Estonia,Europe,Spain,Europe,EE181042894443739087
1,TX001242,168.11,2023-05-19 16:44:53,Credit,New York,D000050,146.69.70.214,M015,INTERNET BANKING,51,...,0.0,2013-08-21,2023-01-26 16:13:15,2013-08-21,Kuressaare,Estonia,Europe,Spain,Europe,EE670595952060641641
2,TX001055,382.07,2023-06-05 17:48:07,Credit,Omaha,D000318,21.97.154.92,M091,BRANCH,62,...,0.0,2013-08-21,2023-05-19 16:44:53,2013-08-21,Rakvere,Estonia,Europe,Qatar,Asia,EE744909036826884237
3,TX001747,305.18,2023-06-05 17:48:33,Debit,Chicago,D000699,93.146.251.20,M065,INTERNET BANKING,58,...,0.0,2013-08-21,2023-06-05 17:48:07,2013-08-21,Yangon,Estonia,Europe,Czech Republic,Europe,EE381758714454105442
4,TX001190,7.95,2023-01-09 16:45:12,Debit,Phoenix,D000147,101.120.142.50,M087,ATM,79,...,0.0,2013-06-29,2013-06-29 00:00:00,2013-06-29,Narva,Estonia,Europe,Singapore,Asia,EE235200952890698989


In [75]:
df_mix.columns

Index(['transaction_id', 'transaction_amount', 'transaction_date',
       'transaction_type', 'location_device', 'device_id', 'ip_address',
       'merchant_id', 'channel', 'customer_age', 'customer_occupation',
       'transaction_duration', 'login_attempts', 'account_balance',
       'previous_transaction_date_raw', 'customer_id', 'account_type',
       'currency', 'account_id', 'first_name', 'surname', 'balance',
       'dt_customer', 'previous_transaction_date', 'account_created_date',
       'location', 'receiver_country', 'receiver_region', 'sender_country',
       'sender_region', 'sender_account'],
      dtype='object')

In [76]:
# re order

new_order = [
    # Transaction info
    'transaction_id', 'transaction_date', 'transaction_amount', 'transaction_type',
    'transaction_duration', 'login_attempts',
    'previous_transaction_date', 'previous_transaction_date_raw',

    # Device & channel
    'location_device', 'device_id', 'ip_address', 'channel', 'location',

    # Customer info
    'customer_id', 'first_name', 'surname', 'dt_customer',

    # Customer account info
    'account_id', 'account_type', 'currency', 'account_created_date',

    # Sender info
    'sender_account', 'sender_country', 'sender_region',

    # Receiver info
    'receiver_country', 'receiver_region',
]

df_mix = df_mix[new_order]


In [77]:
df_mix.head()

,transaction_id,transaction_date,transaction_amount,transaction_type,transaction_duration,login_attempts,previous_transaction_date,previous_transaction_date_raw,location_device,device_id,...,dt_customer,account_id,account_type,currency,account_created_date,sender_account,sender_country,sender_region,receiver_country,receiver_region
0,TX001910,2023-01-26 16:13:15,301.47,Debit,291,1,2013-08-21 00:00:00,2024-11-04 08:09:25,Colorado Springs,D000030,...,2013-08-21,EE816402671097545968,SAVINGS,USD,2013-08-21,EE181042894443739087,Spain,Europe,Estonia,Europe
1,TX001242,2023-05-19 16:44:53,168.11,Credit,12,1,2023-01-26 16:13:15,2024-11-04 08:06:59,New York,D000050,...,2013-08-21,EE816402671097545968,SAVINGS,USD,2013-08-21,EE670595952060641641,Spain,Europe,Estonia,Europe
2,TX001055,2023-06-05 17:48:07,382.07,Credit,147,1,2023-05-19 16:44:53,2024-11-04 08:08:17,Omaha,D000318,...,2013-08-21,EE816402671097545968,SAVINGS,USD,2013-08-21,EE744909036826884237,Qatar,Asia,Estonia,Europe
3,TX001747,2023-06-05 17:48:33,305.18,Debit,216,1,2023-06-05 17:48:07,2024-11-04 08:08:22,Chicago,D000699,...,2013-08-21,EE816402671097545968,SAVINGS,USD,2013-08-21,EE381758714454105442,Czech Republic,Europe,Estonia,Europe
4,TX001190,2023-01-09 16:45:12,7.95,Debit,65,1,2013-06-29 00:00:00,2024-11-04 08:10:06,Phoenix,D000147,...,2013-06-29,EE439018519889093488,BUSINESS,EUR,2013-06-29,EE235200952890698989,Singapore,Asia,Estonia,Europe


In [78]:
# Eksport final data to csv (folder final source)


##df_mix.to_csv(
##    r"D:\KULIAH\SEMESTER 4\DRAFT\Step by step\Final Source Data\TRANSACTION_MASTER_FULL.csv",
##   index=False
##)

df_mix.to_csv("TRANSACTION_MASTER_FULL.csv", index=False)